In [42]:
from google.colab import drive
import zipfile
import os
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/Sentiment_Data.zip"

extract_dir = "/content/sentiment_data"
os.makedirs(extract_dir, exist_ok=True)


with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    print("Files inside ZIP:")
    zip_ref.printdir()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files inside ZIP:
File Name                                             Modified             Size
Sentiment_Data.csv                             2023-06-20 16:16:42     82889405


In [43]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Sentiment_Data.zip"
extract_dir = "/content/sentiment_data"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(" ZIP extracted successfully.")


 ZIP extracted successfully.


In [45]:
import pandas as pd

csv_path = os.path.join(extract_dir, "Sentiment_Data.csv")

try:
    df = pd.read_csv(csv_path, encoding='utf-8')
except UnicodeDecodeError:
    print(" UTF-8 decoding failed. Trying ISO-8859-1...")
    df = pd.read_csv(csv_path, encoding='ISO-8859-1')

print(" Dataset Loaded!")
print("Shape of dataset:", df.shape)

df.head()


 UTF-8 decoding failed. Trying ISO-8859-1...
 Dataset Loaded!
Shape of dataset: (451332, 2)


,Tweet,Sentiment
0,@_angelica_toy Happy Anniversary!!!....The Day...,Mild_Pos
1,@McfarlaneGlenda Happy Anniversary!!!....The D...,Mild_Pos
2,@thevivafrei @JustinTrudeau Happy Anniversary!...,Mild_Pos
3,@NChartierET Happy Anniversary!!!....The Day t...,Mild_Pos
4,@tabithapeters05 Happy Anniversary!!!....The D...,Mild_Pos


In [47]:
print("\n Dataset Columns:")
print(df.columns)

print("\n Label Distribution (if column is named 'label'):")
if 'label' in df.columns:
    print(df['label'].value_counts())




 Dataset Columns:
Index(['Tweet', 'Sentiment'], dtype='object')

 Label Distribution (if column is named 'label'):


In [ ]:
print("Unique Sentiment Labels:")
print(df['Sentiment'].unique())


Unique Sentiment Labels:
['Mild_Pos' 'Strong_Pos' 'Neutral' 'Strong_Neg' 'Mild_Neg']


In [48]:
label_map = {
    'Mild_Pos': 'Positive',
    'Strong_Pos': 'Positive',
    'Neutral': 'Neutral',
    'Mild_Neg': 'Negative',
    'Strong_Neg': 'Negative'
}

df['Sentiment'] = df['Sentiment'].map(label_map)

print(" Labels mapped successfully!")
print(df['Sentiment'].value_counts())


 Labels mapped successfully!
Sentiment
Positive    297704
Neutral      77016
Negative     76612
Name: count, dtype: int64


In [49]:
import re

def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_tweet'] = df['Tweet'].apply(clean_tweet)

print(" Sample cleaned tweets:")
df[['Tweet', 'clean_tweet']].head()


 Sample cleaned tweets:


,Tweet,clean_tweet
0,@_angelica_toy Happy Anniversary!!!....The Day...,happy anniversarythe day the freedumb died in ...
1,@McfarlaneGlenda Happy Anniversary!!!....The D...,happy anniversarythe day the freedumb died in ...
2,@thevivafrei @JustinTrudeau Happy Anniversary!...,happy anniversarythe day the freedumb died in ...
3,@NChartierET Happy Anniversary!!!....The Day t...,happy anniversarythe day the freedumb died in ...
4,@tabithapeters05 Happy Anniversary!!!....The D...,happy anniversarythe day the freedumb died in ...


In [50]:
from sklearn.model_selection import train_test_split

X = df['clean_tweet'].values
y = df['Sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 361065
Testing samples: 90267


In [51]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

MAX_LEN = 100
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post')

print(" Tokenization & Padding complete!")


 Tokenization & Padding complete!


In [52]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

y_train_cat = to_categorical(y_train_enc, num_classes=3)
y_test_cat = to_categorical(y_test_enc, num_classes=3)

print(" Label Encoding complete!")
print("Classes:", label_encoder.classes_)


 Label Encoding complete!
Classes: ['Negative' 'Neutral' 'Positive']


In [54]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dropout, Dense
import time

model = Sequential()
model.add(Embedding(input_dim=20000, output_dim=128))
model.add(Bidirectional(LSTM(64, return_sequences=False)))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dense(3, activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

start = time.time()

history = model.fit(
    X_train_pad, y_train_cat,
    epochs=4,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

end = time.time()
print(" Total Training Time:", round(end - start, 2), "seconds")


Epoch 1/4
2539/2539 ━━━━━━━━━━━━━━━━━━━━ 1225s 480ms/step - accuracy: 0.8180 - loss: 0.4821 - val_accuracy: 0.9084 - val_loss: 0.2571
Epoch 2/4
2539/2539 ━━━━━━━━━━━━━━━━━━━━ 1249s 491ms/step - accuracy: 0.9184 - loss: 0.2363 - val_accuracy: 0.9188 - val_loss: 0.2362
Epoch 3/4
2539/2539 ━━━━━━━━━━━━━━━━━━━━ 1209s 462ms/step - accuracy: 0.9325 - loss: 0.1985 - val_accuracy: 0.9220 - val_loss: 0.2350
Epoch 4/4
2539/2539 ━━━━━━━━━━━━━━━━━━━━ 1158s 456ms/step - accuracy: 0.9417 - loss: 0.1703 - val_accuracy: 0.9207 - val_loss: 0.2457
 Total Training Time: 4842.2 seconds


In [57]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import numpy as np

y_pred_probs = model.predict(X_test_pad)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

cm = confusion_matrix(y_true, y_pred)
print(" Confusion Matrix:\n", cm)

print("\n Classification Report:")
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

auc_score = roc_auc_score(y_test_cat, y_pred_probs, multi_class='ovr')
print(" Macro-Averaged AUC Score:", round(auc_score, 4))


2821/2821 ━━━━━━━━━━━━━━━━━━━━ 106s 37ms/step
 Confusion Matrix:
 [[13103  1379   841]
 [ 1061 12344  1998]
 [  551  1336 57654]]

 Classification Report:
              precision    recall  f1-score   support

    Negative       0.89      0.86      0.87     15323
     Neutral       0.82      0.80      0.81     15403
    Positive       0.95      0.97      0.96     59541

    accuracy                           0.92     90267
   macro avg       0.89      0.87      0.88     90267
weighted avg       0.92      0.92      0.92     90267

 Macro-Averaged AUC Score: 0.9761
